# 04 — Live Dashboard · Colab Edition · *The Sentinel* NIDS

The repo ships a React + Vite command center (`nids-frontend/`) — a Node dev
server can't meaningfully run inside a Colab VM, so this notebook rebuilds
the **same dashboard over the same API** with Gradio:

| React page | This notebook |
|---|---|
| `/` Dashboard — KPIs, attack pie, live alert feed | **📊 Dashboard** tab (auto-refresh) |
| `/reports` + `/network` — distribution, leaderboard, timeline | **🕵 Threat intel** tab |
| `/explain` — SHAP feature contributions per alert | **🧠 Explain** tab |
| attack simulators + `send_attacks.py` replay | **🧪 Traffic lab** tab |
| WebSocket live stream | 3-second polling of the same endpoints |

The notebook boots the **same embedded API** as `03_Inference_API_Colab.ipynb`
(predict → persist → broadcast on SQLite), so the dashboard consumes real
alerts — nothing synthetic.

**Before running:** artifacts from `02_Training_GPU_Colab.ipynb`
(Drive sync or `nids_artifacts.zip` in `/content/`).

In [ ]:
!pip install -q gradio shap

## ⚙️ Step 1 — Config & artifact discovery

The trained artifacts produced by **02_Training_GPU_Colab.ipynb** are the
deployment contract of this notebook (mirrors `model.pkl` / `scaler.pkl` /
`label_encoder.pkl` in the repo). They are searched in this order:

1. `/content/nids_artifacts/` (produced by notebook 02 in this session)
2. Google Drive → `MyDrive/nids_artifacts/` (if notebook 02 synced them)
3. A `nids_artifacts.zip` you uploaded to `/content/` (fallback)

The raw CSV (`DATA_PATH`) is only needed for the traffic-replay section.

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────
DRIVE_ART_DIR   = "nids_artifacts"    # folder inside MyDrive (sync target)
DRIVE_DATA_DIR  = "nids_data"         # folder inside MyDrive (dataset)
DATA_FILENAME   = "cicids2017_cleaned.csv"
API_PORT        = 8000

API_BASE = f"http://127.0.0.1:{API_PORT}"

# ── Locate trained artifacts ─────────────────────────────────────────────
def _find_artifacts() -> Path | None:
    candidates = [
        Path("/content/nids_artifacts"),
        Path("/content/drive/MyDrive") / DRIVE_ART_DIR,
    ]
    for cand in candidates:
        if (cand / "model.pkl").exists():
            return cand
    # optional zip fallback: nids_artifacts.zip in /content
    zpath = Path("/content/nids_artifacts.zip")
    if zpath.exists():
        with zipfile.ZipFile(zpath) as z:
            z.extractall("/content/nids_artifacts")
        return Path("/content/nids_artifacts")
    return None

ART_DIR = _find_artifacts()

if ART_DIR is None:
    # last chance: mount Drive (skipped earlier if it would block)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ART_DIR = _find_artifacts()
    except Exception as e:
        print(f"(Drive mount unavailable: {e})")

assert ART_DIR is not None, (
    "Trained artifacts not found!\n"
    "→ Run 02_Training_GPU_Colab.ipynb first (it saves to /content/nids_artifacts\n"
    "  and syncs to MyDrive/nids_artifacts), or upload nids_artifacts.zip to /content."
)
print(f"Artifacts found: {ART_DIR}")
print("Contents:", sorted(p.name for p in ART_DIR.iterdir()))

# ── Locate the dataset (only needed for the replay section) ──────────────
DATA_PATH = None
for cand in [
    Path("/content") / DATA_FILENAME,
    Path("/content/drive/MyDrive") / DRIVE_DATA_DIR / DATA_FILENAME,
]:
    if cand.exists():
        DATA_PATH = cand
        break

if DATA_PATH is None:
    print(f"\n[!] {DATA_FILENAME} not found — the traffic-replay section will be skipped.")
    print("    Put the CSV in MyDrive/nids_data/ to enable it.")
else:
    print(f"Dataset found: {DATA_PATH}")

## 🧠 Step 2 — Inference stack (port of `src/model/predict.py`)

This cell embeds the exact production inference contract:
- `CICIDS_FEATURES` — the 52-feature list from `src/features/extractor.py`
- `SEVERITY_MAP` / `get_severity()` / `is_benign()` — from `src/model/predict.py`
- `BENIGN_LABELS` — from `src/api/constants.py`
- artifact loading with the `n_features_in_` parity guard
- cached SHAP `TreeExplainer` (created once, not per request)

In [ ]:
import joblib
import numpy as np
import pandas as pd

# ── src/api/constants.py ─────────────────────────────────────────────────
BENIGN_LABELS = ("Normal Traffic", "BENIGN")

# ── src/features/extractor.py::CICIDS_FEATURES (order = model contract) ──
CICIDS_FEATURES = [
    'Destination Port',
    'Flow Duration',
    'Total Fwd Packets',
    'Total Length of Fwd Packets',
    'Fwd Packet Length Max',
    'Fwd Packet Length Min',
    'Fwd Packet Length Mean',
    'Fwd Packet Length Std',
    'Bwd Packet Length Max',
    'Bwd Packet Length Min',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Std',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Flow IAT Mean',
    'Flow IAT Std',
    'Flow IAT Max',
    'Flow IAT Min',
    'Fwd IAT Total',
    'Fwd IAT Mean',
    'Fwd IAT Std',
    'Fwd IAT Max',
    'Fwd IAT Min',
    'Bwd IAT Total',
    'Bwd IAT Mean',
    'Bwd IAT Std',
    'Bwd IAT Max',
    'Bwd IAT Min',
    'Fwd Header Length',
    'Bwd Header Length',
    'Fwd Packets/s',
    'Bwd Packets/s',
    'Min Packet Length',
    'Max Packet Length',
    'Packet Length Mean',
    'Packet Length Std',
    'Packet Length Variance',
    'FIN Flag Count',
    'PSH Flag Count',
    'ACK Flag Count',
    'Average Packet Size',
    'Subflow Fwd Bytes',
    'Init_Win_bytes_forward',
    'Init_Win_bytes_backward',
    'act_data_pkt_fwd',
    'min_seg_size_forward',
    'Active Mean',
    'Active Max',
    'Active Min',
    'Idle Mean',
    'Idle Max',
    'Idle Min',
]

# ── src/model/predict.py::SEVERITY_MAP (21 keys) ─────────────────────────
SEVERITY_MAP = {
    "benign":             "NONE",
    "normal traffic":     "NONE",
    "normal":             "NONE",
    "ddos":               "CRITICAL",
    "dos":                "CRITICAL",
    "dos hulk":           "CRITICAL",
    "dos goldeneye":      "CRITICAL",
    "dos slowloris":      "CRITICAL",
    "dos slowhttptest":   "CRITICAL",
    "heartbleed":         "CRITICAL",
    "bot":                "HIGH",
    "ftp-patator":        "HIGH",
    "ssh-patator":        "HIGH",
    "infiltration":       "HIGH",
    "port scanning":      "MEDIUM",
    "portscan":           "MEDIUM",
    "web attack":         "MEDIUM",
    "web attack – brute force": "MEDIUM",
    "web attack – xss":   "MEDIUM",
    "web attack – sql injection": "MEDIUM",
    "brute force":        "LOW",
}

def get_severity(prediction: str) -> str:
    key = prediction.strip().lower()
    for pattern, sev in SEVERITY_MAP.items():
        if pattern in key:
            return sev
    return "LOW"

def is_benign(prediction: str) -> bool:
    """The model's benign class is 'Normal Traffic' (not 'BENIGN') — this
    helper covers both spellings so stats/broadcast never misclassify."""
    key = prediction.strip().lower()
    return any(label in key for label in ("normal traffic", "benign", "normal"))

# ── Artifact loading with the production parity guard ────────────────────
_model   = None
_scaler  = None
_encoder = None
_explainer = None
_model_loaded = False

def load_artifacts(art_dir: Path):
    """Load model/scaler/encoder and cache a SHAP TreeExplainer.
    Raises if the artifact widths don't match the 52-feature contract —
    silently-corrupted inference is refused (same as production)."""
    global _model, _scaler, _encoder, _explainer, _model_loaded
    _model   = joblib.load(art_dir / "model.pkl")
    _scaler  = joblib.load(art_dir / "scaler.pkl")
    _encoder = joblib.load(art_dir / "label_encoder.pkl")

    n_model  = int(getattr(_model, "n_features_in_", 0))
    n_scaler = int(getattr(_scaler, "n_features_in_", 0)) if hasattr(_scaler, "n_features_in_") else 0
    expected = len(CICIDS_FEATURES)
    if n_model not in (0, expected) or n_scaler not in (0, expected):
        raise RuntimeError(
            f"Artifact/feature mismatch: model expects {n_model}, scaler expects "
            f"{n_scaler}, contract provides {expected}. Refusing to serve."
        )
    try:
        import shap
        _explainer = shap.TreeExplainer(_model)
        print("SHAP TreeExplainer cached.")
    except Exception as e:
        print(f"SHAP explainer init failed (will skip SHAP): {e}")
        _explainer = None
    _model_loaded = True
    print(f"Model loaded: {type(_model).__name__}")
    print(f"Classes     : {list(_encoder.classes_)}")

load_artifacts(ART_DIR)

def predict_flow(features: dict, feature_names: list | None = None) -> dict:
    """Run inference on one flow's 52-feature dict.
    Returns: prediction, confidence, severity, shap_top5 (top-5 for attacks)."""
    if not _model_loaded:
        raise RuntimeError("Model not loaded.")
    names = list(features.keys())
    expected_len = int(getattr(_model, "n_features_in_", len(CICIDS_FEATURES)))
    if len(names) != expected_len:
        raise ValueError(
            f"Feature vector has {len(names)} keys; model expects {expected_len}."
        )
    values = np.array(list(features.values()), dtype=np.float64).reshape(1, -1)
    if not np.isfinite(values).all():
        raise ValueError("Non-finite feature values.")

    values_scaled = _scaler.transform(values)
    pred_index    = int(_model.predict(values_scaled)[0])
    probabilities = _model.predict_proba(values_scaled)[0]
    confidence    = float(probabilities[pred_index])
    prediction    = _encoder.inverse_transform([pred_index])[0]
    severity      = get_severity(prediction)

    shap_top5 = []
    if _explainer is not None and not is_benign(prediction):
        try:
            shap_values = _explainer.shap_values(values_scaled)
            use_names = feature_names or names
            if isinstance(shap_values, list):
                sv = np.array(shap_values[pred_index]).flatten()
            elif hasattr(shap_values, "ndim"):
                if shap_values.ndim == 3:
                    sv = shap_values[0, :, pred_index]
                else:
                    sv = shap_values[0]
            else:
                sv = np.array(shap_values).flatten()
            pairs = sorted(zip(use_names, sv.tolist()),
                           key=lambda x: abs(float(x[1])), reverse=True)[:5]
            shap_top5 = [{"feature": n, "value": round(float(v), 4)} for n, v in pairs]
        except Exception as e:
            print(f"SHAP inference failed: {e}")

    return {
        "prediction": prediction,
        "confidence": round(confidence, 4),
        "severity":   severity,
        "shap_top5":  shap_top5,
    }

print("Inference stack ready ✔")

## 🗄️ Step 3 — Persistence + validation helpers

Ports of:
- `src/api/database.py` (SQLite + WAL pragmas) and `src/api/models.py` (`Alert`)
- `src/api/routes/predict.py` validation (`_validate_features`, `_coerce_port`,
  `_validate_metadata_ip`, `_sanitize_shap`) — no silent coercion, ≤8 missing
  features tolerated, non-negative finite values only, real IPv4/IPv6 metadata.

In [ ]:
import math
import time
import ipaddress
import threading
import asyncio
import json as _json
from collections import defaultdict
from datetime import datetime, timezone

from sqlalchemy import (
    create_engine, event, Column, Integer, String, Float, DateTime, Text,
    desc, func, text as sql_text,
)
from sqlalchemy.orm import declarative_base, sessionmaker

# ── database.py ──────────────────────────────────────────────────────────
DB_URL = f"sqlite:////content/nids_colab.db"
_engine = create_engine(DB_URL, connect_args={"check_same_thread": False}, pool_pre_ping=True)

@event.listens_for(_engine, "connect")
def _sqlite_pragmas(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    try:
        cursor.execute("PRAGMA journal_mode=WAL")
        cursor.execute("PRAGMA busy_timeout=5000")
    finally:
        cursor.close()

SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=_engine)
Base = declarative_base()

def utcnow() -> datetime:
    return datetime.now(timezone.utc)

def iso_utc(dt) -> str | None:
    if dt is None:
        return None
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc).isoformat()

# ── models.py ────────────────────────────────────────────────────────────
class Alert(Base):
    __tablename__ = "alerts"
    id             = Column(Integer, primary_key=True, index=True)
    timestamp      = Column(DateTime, default=utcnow, index=True)
    source_ip      = Column(String(45), index=True)
    destination_ip = Column(String(45))
    src_port       = Column(Integer, nullable=True)
    dst_port       = Column(Integer, nullable=True)
    prediction     = Column(String(50), index=True)
    confidence     = Column(Float)
    severity       = Column(String(20), index=True)
    shap_json      = Column(Text, nullable=True)

Base.metadata.create_all(bind=_engine)
print(f"SQLite ready → {DB_URL}")

# ── routes/predict.py validation ─────────────────────────────────────────
EXPECTED_FEATURES      = list(CICIDS_FEATURES)
MAX_INVALID_FEATURES   = 8
MAX_ABS_FEATURE_VALUE  = 1e15
MAX_METADATA_IP_LENGTH = 45

def _validate_features(raw: dict):
    """Classify each expected feature as valid / missing / invalid.
    Missing → 0.0 (tolerated up to 8); non-numeric, non-finite, negative or
    >1e15 values are invalid and reject the request (no silent coercion)."""
    features, missing, invalid = {}, 0, []
    for feat_name in EXPECTED_FEATURES:
        if feat_name not in raw:
            features[feat_name] = 0.0
            missing += 1
            continue
        try:
            value = float(raw[feat_name])
        except (ValueError, TypeError):
            invalid.append(feat_name)
            features[feat_name] = 0.0
            continue
        if (not math.isfinite(value) or value < 0 or abs(value) > MAX_ABS_FEATURE_VALUE):
            invalid.append(feat_name)
            features[feat_name] = 0.0
            continue
        features[feat_name] = value
    return features, missing, invalid

def _coerce_port(value) -> int:
    try:
        as_float = float(value)
    except (ValueError, TypeError):
        return 0
    if not math.isfinite(as_float):
        return 0
    port = int(as_float)
    return port if 0 <= port <= 65535 else 0

def _validate_metadata_ip(value, field: str) -> str:
    """Metadata IPs must be real IPv4/IPv6 — blocks log forging and
    data-borne prompt injection (same rule as production)."""
    if value is None:
        return "unknown"
    text_value = str(value).strip()
    if not text_value or text_value.lower() == "unknown":
        return "unknown"
    if len(text_value) > MAX_METADATA_IP_LENGTH:
        raise HTTPException(422, detail={"message": f"{field} exceeds max IP length.", "field": field})
    if any(not ch.isprintable() for ch in text_value):
        raise HTTPException(422, detail={"message": f"{field} contains control characters.", "field": field})
    try:
        ipaddress.ip_address(text_value)
    except ValueError:
        raise HTTPException(422, detail={"message": f"{field} must be a valid IPv4/IPv6 address.", "field": field})
    return text_value

def _sanitize_shap(shap_top5: list) -> list:
    """Coerce non-finite SHAP values to 0.0 → browser-safe JSON."""
    safe = []
    for item in shap_top5 or []:
        try:
            value = float(item.get("value", 0.0))
        except (TypeError, ValueError):
            value = 0.0
        if not math.isfinite(value):
            value = 0.0
        safe.append({"feature": str(item.get("feature", "")), "value": round(value, 4)})
    return safe

print("Validation helpers ready ✔")

## 🌐 Step 4 — The FastAPI app (port of `src/api/main.py` + routes)

Endpoints (same contract as the production backend):

| Method | Endpoint | Purpose |
|---|---|---|
| GET | `/health` | liveness: db, model, uptime |
| POST | `/api/predict` | classify one flow (52 CICIDS features + `_source_ip` etc.) |
| GET | `/api/alerts` | paginated alert history (filters: `type`, `severity`, `exclude_benign`) |
| GET | `/api/stats` | totals, attacks by type / severity, uptime |
| GET | `/api/ip-leaderboard` | top attacking source IPs |
| WS | `/ws/live` | live attack broadcast (last-50 history on connect, ping every 10 s) |

> The sniffer endpoints (`/api/sniffer/*`) are **not** portable to Colab — a VM
> has no raw-packet access to your LAN. The offline replay (Step 6) feeds the
> same pipeline instead.

In [ ]:
from fastapi import FastAPI, HTTPException, Request, WebSocket, WebSocketDisconnect
from starlette.concurrency import run_in_threadpool
from typing import List, Optional

_START_TIME = time.time()
RATE_LIMIT_PER_MINUTE = 120
MAX_BODY_BYTES = 1_000_000
_rate_hits: dict = defaultdict(list)
OPEN_PATHS = {"/", "/health", "/docs", "/openapi.json"}


class ConnectionManager:
    """Bounded WebSocket fan-out with per-client send timeout
    (ports main.py::ConnectionManager)."""

    def __init__(self, max_clients: int = 20, send_timeout_s: float = 5.0):
        self.active: List[WebSocket] = []
        self.max_clients = max_clients
        self.send_timeout_s = send_timeout_s

    def can_accept(self) -> bool:
        return len(self.active) < self.max_clients

    async def connect(self, ws: WebSocket):
        await ws.accept()
        self.active.append(ws)

    def disconnect(self, ws: WebSocket):
        if ws in self.active:
            self.active.remove(ws)

    async def _safe_send(self, ws: WebSocket, data: str) -> bool:
        try:
            await asyncio.wait_for(ws.send_text(data), timeout=self.send_timeout_s)
            return True
        except Exception:
            self.disconnect(ws)
            return False

    async def broadcast(self, message: dict):
        if not self.active:
            return
        data = _json.dumps(message)
        await asyncio.gather(*(self._safe_send(ws, data) for ws in list(self.active)))


ws_manager = ConnectionManager()
app = FastAPI(title="NIDS — Network Intrusion Detection API (Colab)", version="2.0.0-colab")


@app.middleware("http")
async def security_middleware(request, call_next):
    path = request.url.path
    if path not in OPEN_PATHS:
        content_length = request.headers.get("content-length", "")
        if content_length.isdigit() and int(content_length) > MAX_BODY_BYTES:
            from fastapi.responses import JSONResponse
            return JSONResponse(status_code=413, content={"detail": "Request body too large."})
        client_ip = request.client.host if request.client else "unknown"
        now = time.time()
        window = [t for t in _rate_hits[client_ip] if t > now - 60]
        window.append(now)
        _rate_hits[client_ip] = window
        if len(window) > RATE_LIMIT_PER_MINUTE:
            from fastapi.responses import JSONResponse
            return JSONResponse(status_code=429, content={"detail": "Rate limit exceeded."})
    return await call_next(request)


@app.get("/health")
def health_check():
    db_ok = False
    try:
        db = SessionLocal()
        try:
            db.execute(sql_text("SELECT 1"))
            db_ok = True
        finally:
            db.close()
    except Exception:
        pass
    return {
        "status": "ok",
        "db": "ok" if db_ok else "error",
        "model": "ok" if _model_loaded else "not loaded",
        "uptime_seconds": round(time.time() - _START_TIME, 1),
        "ws_clients": len(ws_manager.active),
    }


@app.post("/api/predict")
async def predict_flow_route(request: Request):
    try:
        raw = await request.json()
    except Exception:
        raise HTTPException(status_code=400, detail="Invalid JSON payload.")
    if not isinstance(raw, dict):
        raise HTTPException(status_code=400, detail="Body must be a JSON object.")

    source_ip      = _validate_metadata_ip(raw.get("_source_ip"), "_source_ip")
    destination_ip = _validate_metadata_ip(raw.get("_destination_ip"), "_destination_ip")
    src_port       = _coerce_port(raw.get("_src_port", 0))
    dst_port       = _coerce_port(raw.get("_dst_port", 0))

    features, missing_count, invalid_names = _validate_features(raw)
    if missing_count == len(EXPECTED_FEATURES):
        raise HTTPException(status_code=400,
            detail="Invalid request: none of the 52 CICIDS2017 features were provided.")
    if invalid_names:
        raise HTTPException(status_code=422, detail={
            "message": "Feature values must be finite, non-negative numbers within sane bounds.",
            "invalid_features": invalid_names[:20],
        })
    if missing_count > MAX_INVALID_FEATURES:
        raise HTTPException(status_code=422, detail={
            "message": f"{missing_count} of {len(EXPECTED_FEATURES)} features are missing "
                       f"(max tolerated: {MAX_INVALID_FEATURES}).",
            "missing": missing_count,
        })

    try:
        result = await run_in_threadpool(predict_flow, features, EXPECTED_FEATURES)
    except Exception:
        raise HTTPException(status_code=500, detail="Inference failed. Check server logs.")

    prediction, confidence = result["prediction"], result["confidence"]
    severity, shap_top5    = result["severity"], _sanitize_shap(result.get("shap_top5", []))

    def _persist():
        db = SessionLocal()
        try:
            alert = Alert(
                timestamp=utcnow(), source_ip=source_ip, destination_ip=destination_ip,
                src_port=src_port, dst_port=dst_port, prediction=prediction,
                confidence=confidence, severity=severity,
                shap_json=_json.dumps(shap_top5, allow_nan=False),
            )
            db.add(alert)
            db.commit()
            db.refresh(alert)
            return alert
        finally:
            db.close()

    alert = await run_in_threadpool(_persist)

    if not is_benign(prediction):
        await ws_manager.broadcast({
            "id": alert.id,
            "timestamp": iso_utc(alert.timestamp),
            "src_ip": source_ip,
            "source_ip": source_ip,
            "attack_type": prediction,
            "prediction": prediction,
            "severity": severity,
            "confidence": confidence,
            "shap_top5": shap_top5,
            "missing_features": missing_count,
        })

    return {
        "alert_id": alert.id,
        "prediction": prediction,
        "confidence": confidence,
        "severity": severity,
        "source_ip": source_ip,
        "shap_top5": shap_top5,
        "timestamp": iso_utc(alert.timestamp) or "",
        "missing_features": missing_count,
    }


@app.get("/api/alerts")
def get_alerts(
    limit: int = 50, offset: int = 0,
    type: Optional[str] = None, severity: Optional[str] = None,
    exclude_benign: bool = True,
):
    limit = max(1, min(int(limit), 500))
    offset = max(0, int(offset))
    query = SessionLocal().query(Alert).order_by(desc(Alert.timestamp), desc(Alert.id))
    db = query.session
    try:
        if exclude_benign:
            query = query.filter(Alert.prediction.notin_(BENIGN_LABELS))
        if type:
            term = type.strip()[:100].replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_")
            query = query.filter(Alert.prediction.ilike(f"%{term}%", escape="\\"))
        if severity:
            query = query.filter(Alert.severity == severity.upper())
        rows = query.offset(offset).limit(limit).all()
        return [{
            "id": a.id, "timestamp": iso_utc(a.timestamp),
            "source_ip": a.source_ip, "destination_ip": a.destination_ip,
            "src_port": a.src_port, "dst_port": a.dst_port,
            "prediction": a.prediction,
            "confidence": round(a.confidence, 4) if a.confidence else 0.0,
            "severity": a.severity, "shap_json": a.shap_json,
        } for a in rows]
    finally:
        db.close()


@app.get("/api/stats")
def get_stats():
    db = SessionLocal()
    try:
        is_attack = Alert.prediction.notin_(BENIGN_LABELS)
        total_flows   = db.query(func.count(Alert.id)).scalar() or 0
        total_attacks = db.query(func.count(Alert.id)).filter(is_attack).scalar() or 0
        by_type = dict(db.query(Alert.prediction, func.count(Alert.id))
                       .filter(is_attack).group_by(Alert.prediction).all())
        by_sev  = dict(db.query(Alert.severity, func.count(Alert.id))
                       .filter(is_attack).group_by(Alert.severity).all())
        return {
            "total_flows": total_flows,
            "total_attacks": total_attacks,
            "benign_count": total_flows - total_attacks,
            "attacks_by_type": by_type,
            "attacks_by_severity": by_sev,
            "uptime_seconds": round(time.time() - _START_TIME, 1),
        }
    finally:
        db.close()


@app.get("/api/ip-leaderboard")
def ip_leaderboard(limit: int = 10):
    limit = max(1, min(int(limit), 100))
    db = SessionLocal()
    try:
        rows = (db.query(Alert.source_ip,
                         func.count(Alert.id).label("attack_count"),
                         func.max(Alert.timestamp).label("last_seen"))
                .filter(Alert.prediction.notin_(BENIGN_LABELS))
                .group_by(Alert.source_ip)
                .order_by(desc("attack_count"), desc("last_seen"))
                .limit(limit).all())
        return [{
            "rank": i + 1, "source_ip": r.source_ip,
            "attack_count": r.attack_count, "last_seen": iso_utc(r.last_seen),
        } for i, r in enumerate(rows)]
    finally:
        db.close()


@app.websocket("/ws/live")
async def websocket_live(websocket: WebSocket):
    """Live attack stream: last-50 history on connect, then real-time
    broadcasts, ping every 10 s, client cap enforced."""
    if not ws_manager.can_accept():
        await websocket.accept()
        await websocket.close(code=1013, reason="Too many connected clients")
        return
    await ws_manager.connect(websocket)
    try:
        db = SessionLocal()
        try:
            recent = (db.query(Alert)
                      .filter(Alert.prediction.notin_(BENIGN_LABELS))
                      .order_by(desc(Alert.timestamp), desc(Alert.id))
                      .limit(50).all())
        finally:
            db.close()
        if recent:
            history = [{
                "id": a.id, "timestamp": iso_utc(a.timestamp) or "",
                "src_ip": a.source_ip or "unknown",
                "attack_type": a.prediction, "severity": a.severity,
                "confidence": round(a.confidence or 0, 4),
                "shap_top5": _json.loads(a.shap_json) if a.shap_json else [],
            } for a in reversed(recent)]
            await websocket.send_text(_json.dumps(history))
        while True:
            await asyncio.sleep(10)
            await websocket.send_text(_json.dumps({"type": "ping"}))
    except WebSocketDisconnect:
        ws_manager.disconnect(websocket)
    except Exception:
        ws_manager.disconnect(websocket)


@app.get("/")
def root():
    return {
        "message": "NIDS API v2.0 (Colab) — Real-time Network Intrusion Detection",
        "docs": f"{API_BASE}/docs",
        "health": f"{API_BASE}/health",
        "ws": f"ws://<public-url>/ws/live",
    }


print("FastAPI app ready ✔")

## 🚀 Step 5 — Start the API server (background thread)

Uvicorn runs in a daemon thread so the notebook stays interactive.
`/docs` (Swagger UI) will be available on the public URL created later.

In [ ]:
import threading
import uvicorn

_config = uvicorn.Config(app, host="0.0.0.0", port=API_PORT, log_level="warning")
_server = uvicorn.Server(_config)
_server_thread = threading.Thread(target=_server.run, daemon=True)
_server_thread.start()

import time as _time
import requests as rq
_health = None
for _ in range(30):
    _time.sleep(1)
    try:
        _health = rq.get(f"{API_BASE}/health", timeout=2).json()
        break
    except Exception:
        continue
print("Server health:", _health)
assert _health and _health.get("status") == "ok", "API did not come up — check the cell output above."
print(f"API live at {API_BASE}  (Swagger docs at {API_BASE}/docs)")

## 🖥️ Step 1 — Dashboard data layer

Every figure below reads from the API exactly like the React `client.ts`
does — `/api/stats`, `/api/alerts`, `/api/ip-leaderboard` — so this UI is a
faithful (not simulated) view of the backend state.

In [ ]:
import json
import random
from collections import Counter
from datetime import datetime

import matplotlib.pyplot as plt

SEV_ORDER  = ["CRITICAL", "HIGH", "MEDIUM", "LOW", "NONE"]
SEV_COLORS = {"CRITICAL": "#dc2626", "HIGH": "#ea580c", "MEDIUM": "#d97706",
              "LOW": "#2563eb", "NONE": "#6b7280"}
FEED_HEADERS = ["Time (UTC)", "Source", "Type", "Severity", "Conf"]

def _get(path, **params):
    try:
        return rq.get(f"{API_BASE}{path}", params=params or None, timeout=5).json()
    except Exception:
        return None

def kpi_markdown(stats):
    if not stats:
        return "**⚠️ Backend offline** — re-run the server cell above."
    up = stats.get("uptime_seconds", 0)
    return (
        f"| 🌊 Flows analyzed | 🚨 Attacks | ✅ Benign | ⏱ Uptime |\n"
        f"|---|---|---|---|\n"
        f"| **{stats['total_flows']:,}** | **{stats['total_attacks']:,}** "
        f"| **{stats['benign_count']:,}** | **{int(up // 60)}m {int(up % 60)}s** |"
    )

def attack_pie_fig(stats):
    plt.close("all")
    fig, ax = plt.subplots(figsize=(5.2, 4))
    by_type = (stats or {}).get("attacks_by_type") or {}
    if not by_type:
        ax.text(0.5, 0.5, "No attacks yet\n→ use the Traffic Lab tab",
                ha="center", va="center", fontsize=11)
        ax.axis("off")
    else:
        items = sorted(by_type.items(), key=lambda kv: -kv[1])
        ax.pie([v for _, v in items], labels=[k for k, _ in items],
               autopct=lambda p: f"{p:.0f}%" if p >= 5 else "",
               startangle=90, textprops={"fontsize": 9},
               colors=plt.cm.Set2.colors)
        ax.set_title("Attacks by type", fontweight="bold")
    plt.tight_layout()
    return fig

def severity_bar_fig(stats):
    plt.close("all")
    fig, ax = plt.subplots(figsize=(5.2, 4))
    by_sev = (stats or {}).get("attacks_by_severity") or {}
    order = [s for s in SEV_ORDER if s in by_sev]
    if order:
        bars = ax.bar(order, [by_sev[s] for s in order],
                      color=[SEV_COLORS[s] for s in order])
        ax.bar_label(bars, fontsize=9)
        ax.set_yscale("symlog")
    else:
        ax.text(0.5, 0.5, "No attacks yet", ha="center", va="center", fontsize=11)
        ax.axis("off")
    ax.set_title("Attacks by severity", fontweight="bold")
    plt.tight_layout()
    return fig

def feed_dataframe(limit=12):
    alerts = _get("/api/alerts", limit=limit) or []
    rows = [[
        (a["timestamp"][11:19] if a["timestamp"] else ""),
        f"{a['source_ip']}:{a['src_port']}",
        a["prediction"], a["severity"], f"{a['confidence'] * 100:.1f}%",
    ] for a in alerts]
    return pd.DataFrame(rows, columns=FEED_HEADERS)

def leaderboard_dataframe(limit=10):
    rows = _get("/api/ip-leaderboard", limit=limit) or []
    data = [[r["rank"], r["source_ip"], r["attack_count"],
             (r["last_seen"] or "")[11:19]] for r in rows]
    return pd.DataFrame(data, columns=["Rank", "Attacker IP", "Attacks", "Last seen"])

def timeline_fig():
    plt.close("all")
    fig, ax = plt.subplots(figsize=(7, 4))
    alerts = _get("/api/alerts", limit=500) or []
    hours = Counter()
    for a in alerts:
        try:
            hours[datetime.fromisoformat(a["timestamp"]).strftime("%H:00")] += 1
        except (TypeError, ValueError):
            continue
    if not hours:
        ax.text(0.5, 0.5, "No attacks in the last 500 alerts",
                ha="center", va="center", fontsize=11)
        ax.axis("off")
    else:
        labels = sorted(hours)
        ax.bar(labels, [hours[h] for h in labels], color="#dc2626", alpha=0.85)
        ax.set_xlabel("Hour (UTC)")
        ax.set_ylabel("Alerts")
        ax.tick_params(axis="x", rotation=45)
    ax.set_title("Attack timeline (last 500 alerts)", fontweight="bold")
    plt.tight_layout()
    return fig

def alert_choices(limit=50):
    alerts = _get("/api/alerts", limit=limit) or []
    return [f"#{a['id']} · {a['prediction']} · {a['severity']}" for a in alerts]

def explain_alert(choice):
    """SHAP top-5 bar plot for one alert — the /explain page."""
    if not choice:
        return None, "Pick an alert from the dropdown."
    alerts = _get("/api/alerts", limit=50) or []
    target_id = choice.split(" · ")[0].lstrip("#")
    alert = next((a for a in alerts if str(a["id"]) == target_id), None)
    if alert is None:
        return None, "Alert not found in the last 50."
    try:
        items = json.loads(alert["shap_json"]) if alert["shap_json"] else []
    except json.JSONDecodeError:
        items = []
    if not items:
        return None, f"Alert #{alert['id']} ({alert['prediction']}) has no SHAP payload."
    items = sorted(items, key=lambda s: s["value"])
    plt.close("all")
    fig, ax = plt.subplots(figsize=(7, max(1.6, 0.55 * len(items) + 1)))
    ax.barh([s["feature"][:30] for s in items], [s["value"] for s in items],
            color=["#dc2626" if s["value"] > 0 else "#2563eb" for s in items])
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"SHAP — {alert['prediction']} (confidence {alert['confidence'] * 100:.1f}%)",
                 fontweight="bold")
    ax.set_xlabel("SHAP value (pushes toward / away from this attack class)")
    plt.tight_layout()
    details = (f"**#{alert['id']}** · `{alert['source_ip']}` → `{alert['destination_ip']}` · "
               f"**{alert['prediction']}** · severity **{alert['severity']}** · "
               f"confidence **{alert['confidence'] * 100:.1f}%**")
    return fig, details

## 🧪 Step 2 — Traffic injection (the "attack simulator")

Equivalent of `send_attacks.py` / `src/simulation/*`: balanced CICIDS2017
flows are replayed through `POST /api/predict` with synthetic attacker IPs.
The dashboard tabs pick the new alerts up on their next refresh.

In [ ]:
_POOL = None

def _replay_pool():
    """Lazily build a generous balanced pool (30/class) from the CSV once."""
    global _POOL
    if _POOL is not None:
        return _POOL
    if DATA_PATH is None:
        _POOL = pd.DataFrame()
        return _POOL
    from collections import Counter as _C
    got, batches = _C(), []
    want_default = 30
    for chunk in pd.read_csv(DATA_PATH, chunksize=250_000):
        chunk.columns = chunk.columns.str.strip()
        chunk = chunk.replace([np.inf, -np.inf], np.nan).dropna()
        for cls, group in chunk.groupby("Attack Type"):
            want = want_default * 2 if cls == "Normal Traffic" else want_default
            if got[cls] >= want:
                continue
            take = group.sample(n=min(want - got[cls], len(group)), random_state=42)
            batches.append(take)
            got[cls] += len(take)
        if all(got[c] >= (want_default * 2 if c == "Normal Traffic" else want_default)
               for c in got):
            break
    _POOL = pd.concat(batches).reset_index(drop=True) if batches else pd.DataFrame()
    print(f"Replay pool ready: {got}")
    return _POOL

def inject_traffic(per_type=5):
    pool = _replay_pool()
    if pool.empty:
        return ("**CSV not available** — upload `cicids2017_cleaned.csv` to "
                "`MyDrive/nids_data/` to enable traffic injection.")
    parts = []
    for cls, group in pool.groupby("Attack Type"):
        n = per_type * 2 if cls == "Normal Traffic" else per_type
        parts.append(group.sample(n=min(len(group), n),
                                  random_state=random.randint(0, 10 ** 6)))
    batch = pd.concat(parts)
    feature_cols = [c for c in pool.columns if c != "Attack Type"]
    sent = attacks = 0
    for _, row in batch.iterrows():
        payload = {c: float(row[c]) for c in feature_cols}
        payload["_source_ip"] = f"10.0.0.{1 + (sent % 200)}"
        payload["_destination_ip"] = "192.168.1.10"
        try:
            r = rq.post(f"{API_BASE}/api/predict", json=payload, timeout=15)
            if r.status_code == 200:
                sent += 1
                if r.json()["severity"] != "NONE":
                    attacks += 1
        except Exception:
            continue
    return (f"Injected **{sent}** flows → **{attacks}** produced attack alerts.\n\n"
            f"Switch to the **📊 Dashboard** tab — it auto-refreshes every 3 s.")

## 🚀 Step 3 — Launch the command center

Gradio serves the UI on a **public `*.gradio.live` URL** — open it on your
laptop or phone. The Dashboard tab polls the API every 3 seconds (the
notebook-equivalent of the WebSocket feed).

In [ ]:
try:
    import gradio as gr
    HAVE_GRADIO = True
    print(f"Gradio {gr.__version__} ✔")
except Exception as e:
    HAVE_GRADIO = False
    print(f"(gradio unavailable: {e} — the static fallback dashboard will be used)")

In [ ]:
if HAVE_GRADIO:
    def refresh_all():
        stats = _get("/api/stats")
        return (
            kpi_markdown(stats),
            attack_pie_fig(stats),
            severity_bar_fig(stats),
            feed_dataframe(),
            leaderboard_dataframe(),
            timeline_fig(),
            gr.update(choices=alert_choices()),
        )

    REFRESH_OUTPUTS = None  # set below

    with gr.Blocks(title="The Sentinel — NIDS Command Center") as demo:
        gr.Markdown(
            "# 🛡️ The Sentinel — NIDS Command Center (Colab)\n"
            "ML-powered intrusion detection · CICIDS2017 · live alerts · SHAP explainability")
        kpi_md = gr.Markdown()
        with gr.Tabs():
            with gr.Tab("📊 Dashboard"):
                with gr.Row():
                    pie = gr.Plot(label="Attacks by type")
                    sev = gr.Plot(label="Attacks by severity")
                gr.Markdown("### 🚨 Live alert feed")
                feed = gr.DataFrame(headers=FEED_HEADERS, interactive=False)
            with gr.Tab("🕵 Threat intel"):
                with gr.Row():
                    board = gr.DataFrame(label="Top attackers", interactive=False)
                    tl = gr.Plot(label="Attack timeline")
            with gr.Tab("🧠 Explain"):
                with gr.Row():
                    shap_dd = gr.Dropdown(label="Pick an alert (last 50)",
                                          choices=[], interactive=True)
                shap_info = gr.Markdown()
                shap_plot = gr.Plot(label="SHAP top-5 contributions")
                shap_dd.change(explain_alert, inputs=shap_dd,
                               outputs=[shap_plot, shap_info])
            with gr.Tab("🧪 Traffic lab"):
                gr.Markdown(
                    "Replay balanced CICIDS2017 flows through the API — the offline "
                    "equivalent of `send_attacks.py` / the attack simulators.")
                per_type = gr.Slider(1, 25, value=5, step=1,
                                     label="Flows per attack class")
                inject_btn = gr.Button("Inject traffic 🚀", variant="primary")
                inject_log = gr.Markdown()
                inject_btn.click(inject_traffic, inputs=per_type, outputs=inject_log)
        with gr.Row():
            refresh_btn = gr.Button("⟳ Refresh now")

        REFRESH_OUTPUTS = [kpi_md, pie, sev, feed, board, tl, shap_dd]
        refresh_btn.click(refresh_all, outputs=REFRESH_OUTPUTS)
        demo.load(refresh_all, outputs=REFRESH_OUTPUTS, every=3)

    demo.launch(share=True, show_error=True)
else:
    print("Gradio missing — running the static fallback snapshot instead.")

In [ ]:
# Fallback: static dashboard snapshot (no Gradio) — one refresh per run.
if not HAVE_GRADIO:
    stats = _get("/api/stats")
    print(kpi_markdown(stats))
    fig = attack_pie_fig(stats); plt.show()
    fig = severity_bar_fig(stats); plt.show()
    fig = timeline_fig(); plt.show()
    print("\nTop attackers:")
    display(leaderboard_dataframe())
    print("\nLatest alerts:")
    display(feed_dataframe())
    print("\n(pip install gradio, then re-run, for the live auto-refreshing UI)")

## 🖼️ Step 4 — Static dashboard preview (for your report)
Gradio is the live UI; this cell renders the **same data** as static PNGs so
you can embed a "command-center screenshot" in your project report even
without opening the share URL.

In [ ]:
stats = _get("/api/stats")

previews = [
    ("dashboard_attacks_by_type.png", attack_pie_fig(stats)),
    ("dashboard_attacks_by_severity.png", severity_bar_fig(stats)),
    ("dashboard_timeline.png", timeline_fig()),
]
for name, fig in previews:
    fig.savefig(ART_DIR / name, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"saved → {ART_DIR / name}")

print("\nTop attackers (last refresh):")
display(leaderboard_dataframe())
print("\nLatest alert feed (last refresh):")
display(feed_dataframe(10))

## ✅ Wrap-up — what runs where

| Capability | Repo (local machine) | Colab (these notebooks) |
|---|---|---|
| Model training | `src/model/train.py` | ✔ `02_Training_GPU_Colab.ipynb` (T4 GPU) |
| Inference + SHAP + severity | `src/model/predict.py` | ✔ notebooks 03/04 |
| Flow extraction from packets | `src/features/extractor.py` | ✔ notebook 03 (synthetic flows) |
| REST API + WebSocket | `src/api/*` | ✔ notebooks 03/04 (embedded server) |
| Dataset replay | `send_attacks.py` | ✔ notebooks 03/04 |
| Live dashboard | React app (`npm run dev`) | ✔ this notebook (Gradio UI) |
| **Scapy live capture** | `src/capture/sniffer.py` | ✖ needs raw sockets on a real NIC |
| **Packet simulators** | `src/simulation/*` | ✖ need a real interface to attack |
| Gemini chatbot endpoint | `/api/chat` (LangChain) | ✔ mini-port in notebook 03 (optional) |

**Going back to the local project:** download `nids_artifacts.zip` from
notebook 02, unzip into `nids-backend/`, and run
`uvicorn src.api.main:app --port 8000` + `npm run dev` as usual — the
artifacts are contract-compatible (`model.pkl` / `scaler.pkl` /
`label_encoder.pkl` / `manifest.json`).